In [1]:
import json
import random
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import os
import re
import numpy as np
import base64
import time
import pandas as pd 
from tqdm import tqdm
from word2number import w2n

# Load dataset

In [ ]:
# Set base directory using relative path
base_dir = os.path.join(os.getcwd(), "dataset", "simpsons")

# Set paths relative to base_dir
annotation_path = os.path.join(base_dir, "v1_Annotation_Val_simpsons_vqa.json")
question_path = os.path.join(base_dir, "v1_Question_Val_simpsons_vqa.json")
images_dir = os.path.join(base_dir, "val_images")

def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select the dataset where 'overall_scores' == 1.0
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: annotation['answer']
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [
            question for question in questions
            if question['id'] in question_id_to_answer
        ]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.005, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_correct_answers = [
            question_id_to_answer[q['id']] 
            for q in sampled_questions
        ]
        # Add image_base64 to each sampled question
        for q in sampled_questions:
            image_relative_path = q['img_path']
            image_path = os.path.join(images_dir, image_relative_path)
            q['image_base64'] = encode_image(image_path)

        return sampled_questions, sampled_correct_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            image_base64 = base64.b64encode(image_file.read()).decode('utf-8')
            return image_base64

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

results_single = []


# Single agent prediction

In [3]:
load_dotenv()

# Configuration
MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "claude-3-5-haiku-20241022"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

def get_predict(question, image_base64, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon analysis expert, answer the question strictly based on the visual content and available context using EXACTLY ONE WORD:

    Input Question: {question}

    Guidelines:
    1. Analyze key cartoon elements including character design, facial expressions, compositional framing, color palette, and scene semantics.
    2. No explanations or punctuation allowed.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url",
                                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {"type": "base64",
                                        "media_type": "image/jpeg",
                                        "data": image_base64
                                        }},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1,
                )
                response = completion.content[0].text.strip().lower()

            # Process the response
            # 1. Remove punctuation marks
            response = response.rstrip('.!?')

            # 2. Split into words and get the first word
            words = response.split()
            if not words:
                continue

            first_word = words[0]

            # 3. Convert numbers
            try:
                # Check if word represents a number
                number = w2n.word_to_num(first_word)
                return str(number)
            except ValueError:
                # Check if contains numeric digits
                matches = re.findall(r'\d+', first_word)
                if matches:
                    return matches[0]

                # If not number-related, return first word as is
                return first_word

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print(f"All {max_retries} attempts failed for question: {question}")
    return None


Using OpenAI model: gpt-4o-mini


# Calculate accuracy

In [4]:
def compute_accuracy(question, correct_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2, num_evaluations=3):
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0, [1.0] * num_evaluations

    if correct_answer.lower().strip() + 's' == predicted_answer.lower().strip() or predicted_answer.lower().strip() + 's' == correct_answer.lower().strip():
        return 0.75, [0.75] * num_evaluations
    
    scores = []
    for evaluation_attempt in range(num_evaluations):
        prompt = f"""
        Evaluate the accuracy of the predicted answer according to criteria below:

        Input:
        Question: {question}
        True answer: {correct_answer}
        Predicted answer: {predicted_answer}
        Answer type: {answer_type}

        Evaluation Rules:
        1. Focus on whether the predicted answer correctly captures the core information required by the question.
        2. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].
        3. Do NOT include any explanation, words, punctuation, or additional content.
        4. Answer Type Considerations:
        - Yes/No questions: Check if the meaning is equivalent
        - Number questions: Verify numerical accuracy
        - Other questions: Check for key information match

        Scoring Criteria:
        - 1.0: Contains the correct answer with the same core meaning, even if phrased differently or with additional details
        - 0.75: Mostly correct with only minor differences that slightly change the meaning
        - 0.5: Partially correct - contains some correct elements but misses important aspects
        - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
        - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering by claiming insufficient information
        """

        for attempt in range(max_retries):
            try:
                if is_openai_model:
                    completion = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.choices[0].message.content.strip()
                else:
                    completion = client.messages.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": prompt
                        }],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.content[0].text.strip()

                numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
                if numeric_match:
                    score = float(numeric_match.group(1))
                else:
                    score = 0.0

                scores.append(score)
                break

            except Exception as e:
                print(f"Evaluation attempt {evaluation_attempt+1}, retry {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    continue

    # If all evaluations failed, return 0
    if not scores:
        return 0.0, []
        
    # Calculate the result using majority voting
    from collections import Counter
    vote_counter = Counter(scores)
    majority_score, count = vote_counter.most_common(1)[0]  # Get most common score
    
    # If there's a tie, calculate average of the tied values
    if len(scores) > 2:  # Only check for ties with more than 2 scores
        top_scores = vote_counter.most_common()
        if len(top_scores) > 1 and top_scores[0][1] == top_scores[1][1]:  # If there's a tie
            # Find all scores with the same count
            tied_scores = [score for score, count in top_scores if count == top_scores[0][1]]
            majority_score = sum(tied_scores) / len(tied_scores)  # Average of tied scores
    
    return majority_score, scores

# Evaluate model performance

In [ ]:
try:
    # Initialize counters
    correct_count = 0
    total_count = 0
    # Initialize results storage
    accuracies = []

    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data 
    sampled_questions, sampled_correct_answers = get_dataset(questions, question_id_to_answer)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")


    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set()

    # Process each question
    for question_item, correct_answer in tqdm(zip(sampled_questions, sampled_correct_answers),
                                     total=len(sampled_questions)):
        try:
            question_id = question_item['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue

            processed_question_ids.add(question_id)

            question = question_item['question']
            image_relative_path = question_item['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            image_path = os.path.join(images_dir, image_relative_path)

            model_answer = get_predict(question, question_item['image_base64'])
            pred_solutions = [model_answer] if model_answer is not None else []

            if not pred_solutions:
                print(f"No prediction obtained for question ID {question_id}")
                continue

            accuracy, scores = compute_accuracy(
                question=question,
                correct_answer=correct_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Update counters for percentage-based accuracy
            total_count += 1
            if accuracy >= 0.75:  # Consider it correct if score is 0.75 or higher
                correct_count += 1

            # Store result
            result = {
                'question_id': question_id,
                'question': question,
                'answer_type': answer_type,
                'correct_answer': correct_answer,
                'predicted_answer': model_answer,
                'evaluator_scores': ", ".join([str(s) for s in scores]) if scores else "",  
                'accuracy': accuracy
            }
            results_single.append(result)
            accuracies.append(accuracy)

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question}")
            print(f"Answer Type: {answer_type}")
            print(f"Correct Answer: {correct_answer}")
            if model_answer is not None:
                print(f"Predicted Answer: {model_answer}")
            if scores:
                print(f"Evaluator Scores: {scores}")
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question}")

        except Exception as e:
            print(f"Error processing question {question_item.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy two ways
    average_accuracy_mean = np.mean(accuracies) if accuracies else 0
    average_accuracy_ratio = correct_count / total_count if total_count > 0 else 0
    
    print(f"Average Accuracy (correct/total): {average_accuracy_ratio:.4f}")
    
    # Use the ratio method as the final average accuracy
    average_accuracy = average_accuracy_ratio

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

  3%|▎         | 1/36 [00:02<01:28,  2.52s/it]

Question ID: 77311
Question: what is on the shelf?
Answer Type: other
Correct Answer: book
Predicted Answer: books
Evaluator Scores: [0.75, 0.75, 0.75]
Accuracy: 0.7500


  6%|▌         | 2/36 [00:03<00:50,  1.50s/it]

Question ID: 12809
Question: how many people are in the picture?
Answer Type: number
Correct Answer: 1
Predicted Answer: 1
Evaluator Scores: [1.0, 1.0, 1.0]
Accuracy: 1.0000


  8%|▊         | 3/36 [00:05<01:03,  1.91s/it]

...

Question ID: 71583
Question: what is in the background?
Answer Type: other
Correct Answer: table
Predicted Answer: gameboard
Evaluator Scores: [0.25, 0.25, 0.25]
Accuracy: 0.2500


 97%|█████████▋| 35/36 [01:28<00:03,  3.46s/it]

Question ID: 37083
Question: what are the men doing?
Answer Type: other
Correct Answer: standing
Predicted Answer: discussing
Evaluator Scores: [0.25, 0.25, 0.25]
Accuracy: 0.2500


100%|██████████| 36/36 [01:35<00:00,  2.66s/it]

Question ID: 96818
Question: what is the person sitting on?
Answer Type: other
Correct Answer: chair
Predicted Answer: chair
Evaluator Scores: [1.0, 1.0, 1.0]
Accuracy: 1.0000
Average Accuracy (correct/total): 0.6944


# Save results

In [ ]:
# Clean up evaluation results to remove any existing average rows
results_single = [r for r in results_single if r['question_id'] != 'Average']

# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in results_single))

# Add row numbers to each result
for i, result in enumerate(results_single, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_single) + 1,
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'correct_answer': '',
    'predicted_answer': '',
    'evaluator_scores': '',
    'accuracy': average_accuracy 
}
results_single.append(average_result)

# Define column order with row_num first
column_order = [
    'row_num',
    'question_id',
    'question',
    'answer_type',
    'correct_answer',
    'predicted_answer',
    'evaluator_scores',
    'accuracy'
]

# Create safe model name for file
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Save to CSV
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
os.makedirs(os.path.join(results_dir, "single"), exist_ok=True)

output_path = os.path.join(results_dir, "single", f'simpsons_single_agent_{safe_model_name}.csv')

# First, check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(results_single)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/simpsons_single_agent_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/simpsons_single_agent_gpt_4o_mini.csv
